In [1]:
import os
import pandas as pd

# Odczyt datasetu

In [2]:
df = pd.read_csv(os.path.join("../data", "domy.csv"))

# Usuwanie kolumn

# Zamiana kilku danych tekstowych na numeryczne

In [3]:
# del_cols = ["Id", "Street", "Utilities", "Condition2", "PoolQC", "PoolArea", "LowQualFinSF", "3SsnPorch"]
# df = df.copy().drop(del_cols, axis=1)

In [4]:
df = df.copy()
df["LotFrontage"] = pd.to_numeric(df["LotFrontage"], errors="coerce")
df["MasVnrArea"] = pd.to_numeric(df["MasVnrArea"], errors="coerce")
df["GarageYrBlt"] = pd.to_numeric(df["GarageYrBlt"], errors="coerce")

# Podział danych na zbiór treningowy i testowy

In [5]:
from sklearn.model_selection import train_test_split

X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Inicjalizacja modelu i reportera do treningów

In [6]:
from Regression.training_reporter import TrainingReporter
from Regression.training_model import CustomXGBRegressorModel

model = CustomXGBRegressorModel()
reporter = TrainingReporter(model, X_train, X_test, y_train, y_test)

# Kroswalidacja

In [7]:
reporter.run_cross_validation(cv=5)

Start cross validation...
Fold 0: RMSE = 24155.802284337402, MSE = 583502784.0, R^2 = 0.9097277522087097
Fold 1: RMSE = 34974.60015496961, MSE = 1223222656.0, R^2 = 0.7910206913948059
Fold 2: RMSE = 32959.51844308409, MSE = 1086329856.0, R^2 = 0.8387012481689453
Fold 3: RMSE = 22274.10765889399, MSE = 496135872.0, R^2 = 0.9090991616249084
Fold 4: RMSE = 21660.60608570314, MSE = 469181856.0, R^2 = 0.9110611081123352
RMSE for cross validation: 27204.926925397645 +- 5618.412211984293
MSE for cross validation: 771674604.8 +- 318036369.03443915
R^2 for cross validation: 0.8719219923019409 +- 0.04897331133716214
Full training:
Cross validation finished!
---------------------------------------------------
Model saved in: saved_models\CV_Model.pkl


# Grid search

In [8]:
reporter.run_grid_search(cv=5)

Start grid search...
Grid finished!
Best params: {'model__colsample_bytree': 0.6, 'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__min_child_weight': 2, 'model__n_estimators': 2000, 'model__subsample': 0.8}
Best RMSE score: 26304.0015625
Best MSE score: 715156953.6
Best R^2 score: -0.8786918520927429
---------------------------------------------------
Model saved in: saved_models\GS_Model.pkl


# Randomized grid search

In [9]:
random_grid = reporter.run_randomized_search(cv=5)

Start randomized grid search...
Randomized search finished!
Best params: {'model__subsample': 0.8, 'model__reg_lambda': 5.0, 'model__reg_alpha': 0.6, 'model__n_estimators': np.int64(1300), 'model__min_child_weight': np.int64(6), 'model__max_depth': np.int64(3), 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.6}
Best RMSE score: 26534.913671875
Best MSE score: 725382156.8
Best R^2 score: -0.8785567879676819
---------------------------------------------------
Model saved in: saved_models\RGS_Model.pkl


# Zapis zbioru testowego

In [10]:
reporter.save_test_set()

Test set saved in: saved_models\test_set.csv
